# 05b — Pseudobulk validation

Equivalent to `scripts/05b_pseudobulk_check.py`. Not in the project guide — added
because it is the honest answer to the main statistical criticism of notebook 05.

## The problem

Notebook 05 ran Wilcoxon across ~80,000 cells, a test that assumes cells are
independent replicates. They are not: all cells in `Male_Cocaine_1` came from one
pool of 20 brains processed together. Treating them as 10,000 independent
observations is **pseudoreplication**, and it is the single most-cited criticism of
scRNA-seq differential expression.

## The check

Sum counts per sample → one profile per biological replicate → test 2 vs 2.
With n=2 this is badly underpowered, so **few significant genes is expected and is
not a failure**. What we check is whether the *direction* of change agrees.
High concordance means the cell-level result reflects real biology.

Being able to say this in your Discussion and your rebuttal shows you understand
the statistics rather than just the API.

In [ ]:
import sys

from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'scripts'))



import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import scanpy as sc

import anndata as ad



import config



sc.settings.verbosity = 3

sc.settings.figdir = config.FIG_DIR

sc.settings.set_figure_params(dpi=100, facecolor='white', frameon=False)

sc.logging.print_header()

In [ ]:
from scipy import stats

adata = sc.read_h5ad(config.H5AD_ANNOTATED)
print(f'{adata.n_obs:,} cells')

## Build the pseudobulk matrix

In [ ]:
counts = adata.layers['counts'] if 'counts' in adata.layers else adata.raw.X
genes = adata.var_names if 'counts' in adata.layers else adata.raw.var_names

rows, meta = [], []
for sample in adata.obs['sample'].cat.categories:
    mask = (adata.obs['sample'] == sample).values
    if mask.sum() == 0: continue
    rows.append(np.asarray(counts[mask].sum(axis=0)).ravel())
    o = adata.obs.loc[mask].iloc[0]
    meta.append({'sample': sample, 'sex': o['sex'],
                 'treatment': o['treatment'], 'n_cells': int(mask.sum())})

pb = pd.DataFrame(np.vstack(rows), index=[m['sample'] for m in meta], columns=genes)
meta = pd.DataFrame(meta).set_index('sample')
meta

In [ ]:
pb = pb.loc[:, pb.sum(axis=0) >= 50]   # drop negligible genes
lcpm = np.log2(pb.div(pb.sum(axis=1), axis=0) * 1e6 + 1)
print(f'{pb.shape[1]:,} genes kept')

## Test, 2 vs 2 per sex

In [ ]:
def bh(p):
    p = np.nan_to_num(p, nan=1.0)
    order = np.argsort(p)
    ranked = p[order] * len(p) / (np.arange(len(p)) + 1)
    out = np.empty_like(ranked)
    out[order] = np.minimum.accumulate(ranked[::-1])[::-1].clip(max=1.0)
    return out

pb_results = {}
for sex in ['Male', 'Female']:
    idx = meta.index[meta['sex'] == sex]
    coc = [s for s in idx if meta.loc[s, 'treatment'] == 'Cocaine']
    suc = [s for s in idx if meta.loc[s, 'treatment'] == 'Sucrose']
    a, b = lcpm.loc[coc], lcpm.loc[suc]
    lfc = a.mean(axis=0) - b.mean(axis=0)
    t, p = stats.ttest_ind(a.values, b.values, axis=0, equal_var=True)
    df = pd.DataFrame({'names': lcpm.columns, 'log2FC_pseudobulk': lfc.values,
                       'pval': p, 'pvals_adj': bh(p)}).sort_values('pval')
    df.to_csv(config.TABLE_DIR / f'pseudobulk_de_{sex.lower()}.csv', index=False)
    pb_results[sex] = df
    n_sig = int(((df.pvals_adj < 0.05) & (df.log2FC_pseudobulk.abs() > 1)).sum())
    print(f'{sex}: {n_sig} significant at pseudobulk level (n=2 vs 2)')

print()
print('A small number here is EXPECTED. n=2 has almost no power.')
print('The concordance check below is the informative part.')

## Concordance with the cell-level result

Interpretation:
- **agreement > 80%** → the cell-level result is directionally trustworthy
- **agreement ≈ 50%** → the signal is likely a pseudoreplication artefact

In [ ]:
for sex in ['Male', 'Female']:
    cell = pd.read_csv(config.TABLE_DIR / f'de_global_{sex.lower()}_all.csv')
    merged = cell.merge(pb_results[sex], on='names', suffixes=('_cell', '_pb'))
    hits = merged[(merged.pvals_adj_cell < config.PADJ_THRESHOLD) &
                  (merged.logfoldchanges.abs() > config.LOG2FC_THRESHOLD)]
    if len(hits) < 5:
        print(f'{sex}: too few hits'); continue

    agree = (np.sign(hits.logfoldchanges) == np.sign(hits.log2FC_pseudobulk)).mean()
    rho, prho = stats.spearmanr(hits.logfoldchanges, hits.log2FC_pseudobulk)
    print(f'{sex}: {len(hits)} hits | direction agreement {agree:.1%} | rho={rho:.3f} (p={prho:.2e})')

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(hits.logfoldchanges, hits.log2FC_pseudobulk, s=14, alpha=.7)
    ax.axhline(0, lw=.5, c='k'); ax.axvline(0, lw=.5, c='k')
    ax.set_xlabel('log2FC (cell-level Wilcoxon)')
    ax.set_ylabel('log2FC (pseudobulk, n=2v2)')
    ax.set_title(f'{sex}: rho = {rho:.2f}', fontsize=10)
    fig.tight_layout()
    fig.savefig(config.FIG_DIR / f'05b_pseudobulk_concordance_{sex.lower()}.png', dpi=150)
    plt.show()